In [1]:
!pip install pyspark==3.3.2

# Abaixo temos o versionamento do Java e do Python utilizados para a execução do código. O PySpark requer o Java para funcionar, e é importante garantir que ambos estejam configurados corretamente.

# !java -version
# openjdk version "10.0.2" 2018-07-17
# OpenJDK Runtime Environment 18.3 (build 10.0.2+13)
# OpenJDK 64-Bit Server VM 18.3 (build 10.0.2+13, mixed mode)

# import sys
# print(sys.executable)
# print(sys.version)
# 3.9.13 (main, Aug 25 2022, 23:51:50) [MSC v.1916 64 bit (AMD64)]

Defaulting to user installation because normal site-packages is not writeable


In [10]:
# Executa a sintetização de dados sintéticos a partir de tabelas exemplo

from pyspark.sql import functions as F
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("exploracao_patry").getOrCreate()

import sys
sys.path.insert(0, "../")
from bootstrap_topologico import run_synthesis_from_tables

# Carrega as tabelas via Spark
instrumento = spark.read.options(header=True, inferSchema=True, sep=",").csv("../data_csv/instrumento.csv")
operacao    = spark.read.options(header=True, inferSchema=True, sep=",").csv("../data_csv/operacao.csv")
tipo_if     = spark.read.options(header=True, inferSchema=True, sep=",").csv("../data_csv/tipo_if.csv")

tables = {
    "tipo_if":     tipo_if,
    "instrumento": instrumento,
    "operacao":    operacao,
}

# Specs: PKs e relacionamentos entre as tabelas
specs_config = {
    "tipo_if": {
        "pk_cols": ["NUM_TIPO_IF"],
        "static": True,  
    },
    "instrumento": {
        "pk_cols": ["NUM_IF"],
        "foreign_keys": [
            {
                "columns":        ["NUM_TIPO_IF"],
                "parent_table":   "tipo_if",
                "parent_columns": ["NUM_TIPO_IF"],
            }
        ],
    },
    "operacao": {
        "pk_cols": ["NUM_ID_OPERACAO"],
        "foreign_keys": [
            {
            "columns":        ["NUM_IF", "COD_IF"],
            "parent_table":   "instrumento",
            "parent_columns": ["NUM_IF", "COD_IF"],
            }
        ],
    },
}

# Executa a sintetização
synthetic = run_synthesis_from_tables(
    tables=tables,
    specs_config=specs_config,
    verbose=True,
    scale_factor=10,
)

# Exibe amostras dos dados sintéticos gerados
for nome, df in synthetic.items():
    print(f"\n=== {nome} ===")
    df.show(2, truncate=False)


Ordem topológica: tipo_if -> instrumento -> operacao
n_rows_by_table: {'tipo_if': 5, 'instrumento': 10000, 'operacao': 500000}
Specs ativas após saneamento de relacionamentos:
  tipo_if: sem FK ativa
  OK: instrumento.['NUM_TIPO_IF'] -> tipo_if.['NUM_TIPO_IF']
  OK: operacao.['NUM_IF', 'COD_IF'] -> instrumento.['NUM_IF', 'COD_IF']
[tipo_if] STATIC | 5 linhas
[instrumento] PAI | 1000->10000
[operacao] FILHO | 50000->500000
Validando...
Validação OK.

=== tipo_if ===
+-----------+-----------+--------------------------------+-------------------+-------------------+
|NUM_TIPO_IF|COD_TIPO_IF|NOM_TIPO_IF                     |DAT_INCLUSAO       |DAT_ALTERACAO      |
+-----------+-----------+--------------------------------+-------------------+-------------------+
|1          |CDB        |Certificado de Depósito Bancário|2019-12-01 00:00:00|2019-12-01 00:00:00|
|2          |LCI        |Letra de Crédito Imobiliário    |2019-12-01 00:00:00|2019-12-01 00:00:00|
+-----------+-----------+----------

In [11]:
# Visualização dos dados originais para comparação

tipo_if.show(10,False)
instrumento.filter(F.col('num_if').isin([384,260,694,70])).show(10,False)
operacao.show(4,False)

+-----------+-----------+--------------------------------------+-------------------+-------------------+
|NUM_TIPO_IF|COD_TIPO_IF|NOM_TIPO_IF                           |DAT_INCLUSAO       |DAT_ALTERACAO      |
+-----------+-----------+--------------------------------------+-------------------+-------------------+
|1          |CDB        |Certificado de Depósito Bancário      |2019-12-01 00:00:00|2019-12-01 00:00:00|
|2          |LCI        |Letra de Crédito Imobiliário          |2019-12-01 00:00:00|2019-12-01 00:00:00|
|3          |LCA        |Letra de Crédito do Agronegócio       |2019-12-01 00:00:00|2019-12-01 00:00:00|
|4          |LF         |Letra Financeira                      |2019-12-01 00:00:00|2019-12-01 00:00:00|
|5          |DPGE       |Depósito a Prazo com Garantia Especial|2019-12-01 00:00:00|2019-12-01 00:00:00|
+-----------+-----------+--------------------------------------+-------------------+-------------------+

+------+---------+------------+-----------+-----------

In [12]:
# Visualização dos dados sintetizados para comparação

synthetic['tipo_if'].show(10,False)
synthetic['instrumento'].filter(F.col('num_if').isin([1001,1002,1003,1004])).show(10,False)
synthetic['operacao'].filter(F.col('num_if').isin([1001,1002,1003,1004])).show(10,False)

+-----------+-----------+--------------------------------------+-------------------+-------------------+
|NUM_TIPO_IF|COD_TIPO_IF|NOM_TIPO_IF                           |DAT_INCLUSAO       |DAT_ALTERACAO      |
+-----------+-----------+--------------------------------------+-------------------+-------------------+
|1          |CDB        |Certificado de Depósito Bancário      |2019-12-01 00:00:00|2019-12-01 00:00:00|
|2          |LCI        |Letra de Crédito Imobiliário          |2019-12-01 00:00:00|2019-12-01 00:00:00|
|3          |LCA        |Letra de Crédito do Agronegócio       |2019-12-01 00:00:00|2019-12-01 00:00:00|
|4          |LF         |Letra Financeira                      |2019-12-01 00:00:00|2019-12-01 00:00:00|
|5          |DPGE       |Depósito a Prazo com Garantia Especial|2019-12-01 00:00:00|2019-12-01 00:00:00|
+-----------+-----------+--------------------------------------+-------------------+-------------------+

+------+---------+------------+-----------+-----------